# Built-in Agent Middleware in LangChain

In LangChain v1.x, **Agent Middleware** is a powerful interception layer designed to control, modify, and monitor the execution loop of agents created with `create_agent`.

### What is Agent Middleware?
Unlike **Callbacks** (which are read-only and designed for observability and tracing, like LangSmith), **Middleware** is interactive and sequential. It functions similarly to web server middleware (e.g., in FastAPI or Express), allowing you to:
1. **Intercept and Modify State**: Inspect or alter the `AgentState` before/after LLM calls or tool executions.
2. **Wrap Actions**: Implement retries, caching, or short-circuiting logic on tools or models.
3. **Enforce Safety & Guardrails**: Automatically block, mask, or redact PII (Personally Identifiable Information) before it reaches the model.
4. **Budget & Run Controls**: Set hard limits on the number of model or tool calls to prevent runaway infinite loops.

### Built-in Middleware Components
LangChain comes with several robust, production-ready middlewares out of the box inside `langchain.agents.middleware`:
- `ToolRetryMiddleware`: Automatically retries failing tool executions with customizable backoff, jitter, and error-filtering.
- `ModelRetryMiddleware`: Retries failed model API calls.
- `PIIMiddleware`: Redacts, masks, hashes, or blocks standard PII (emails, URLs, credit cards, IPs) or custom patterns.
- `ModelCallLimitMiddleware` / `ToolCallLimitMiddleware`: Sets maximum execution caps for model and tool calls.
- `SummarizationMiddleware`: Summarizes conversation history/context before the model runs to manage token budgets.

## 1. Setup & Environment

Let's import our environment variables and initialize our chat model using the unified `init_chat_model` function.

In [7]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# Load environment variables (.env)
load_dotenv()

# Initialize the language model
llm = init_chat_model("groq:qwen/qwen3-32b")

## 2. Automatic Tool Retries (`ToolRetryMiddleware`)

In production agents, external tools (APIs, databases, web search) are often flaky or rate-limited. Implementing retry logic at the application level can be complex.

LangChain's `ToolRetryMiddleware` intercepts tool calls, catches exceptions, and retries execution with exponential backoff and jitter.

In [8]:
import random
from langchain_core.tools import tool

# A counter to track tool calls
call_count = 0

@tool
def get_stock_price(ticker: str) -> str:
    """Fetches the current stock price for a given ticker symbol."""
    global call_count
    call_count += 1
    
    print(f"[Tool] Attempt {call_count}: Fetching price for {ticker}...")
    
    # Flaky behavior: Fail the first two times, succeed on the third!
    if call_count < 3:
        raise RuntimeError("Connection timeout while reaching stock service API.")
        
    return f"The price of {ticker} is $240.50"

### Wiring up the Agent with `ToolRetryMiddleware`

We can configure `ToolRetryMiddleware` with:
- `max_retries`: The maximum number of attempts.
- `initial_delay`: The initial wait time in seconds.
- `backoff_factor`: Multiplier for exponential backoff.
- `jitter`: Whether to apply randomized delay variation.

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware

# Reset counter
call_count = 0

# Initialize our retry middleware
retry_middleware = ToolRetryMiddleware(
    max_retries=3,
    initial_delay=1.0,
    backoff_factor=1.5,
    jitter=False
)

# Compile the agent, adding the middleware
agent_with_retry = create_agent(
    model=llm,
    tools=[get_stock_price],
    middleware=[retry_middleware]
)

# Run the agent
result = agent_with_retry.invoke({
    "messages": [{"role": "user", "content": "What is the stock price of MSFT?"}]
})

print("\n[Final Agent Answer]:")
print(result["messages"][-1].content)

[Tool] Attempt 1: Fetching price for MSFT...
[Tool] Attempt 2: Fetching price for MSFT...
[Tool] Attempt 3: Fetching price for MSFT...

[Final Agent Answer]:
The current stock price for Microsoft (MSFT) is **$240.50**. Let me know if you'd like further details!


## 3. PII Redaction & Safety Guardrails (`PIIMiddleware`)

For compliance and security (GDPR, HIPAA, SOC 2), it is crucial to ensure that users do not send sensitive personal information to external LLM providers.

`PIIMiddleware` automatically scans strings and can:
- `redact`: Replace the match with `[REDACTED_EMAIL]`, `[REDACTED_CREDIT_CARD]`, etc.
- `mask`: Replace all but the last few characters.
- `block`: Throw a `PIIDetectionError` instantly.
- `hash`: Deterministically replace with a secure hash value.

You can apply it to user messages (`apply_to_input`), model responses (`apply_to_output`), or tool inputs/outputs (`apply_to_tool_results`).

In [6]:
from langchain.agents.middleware import PIIMiddleware

# Configure PII Redaction Middleware for Email addresses
email_guardrail = PIIMiddleware(
    pii_type="email",
    strategy="redact",
    apply_to_input=True # Intercept and clean user message before model call
)

# Create our agent
safe_agent = create_agent(
    model=llm,
    middleware=[email_guardrail]
)

# Let's invoke the agent with an email in the prompt
prompt = "My name is Alice and my personal email address is alice.secret@example.com. Say hello!"
print(f"Original prompt: {prompt}\n")

result = safe_agent.invoke({
    "messages": [{"role": "user", "content": prompt}]
})

print("[Agent Response]:")
print(result["messages"][-1].content)

Original prompt: My name is Alice and my personal email address is alice.secret@example.com. Say hello!

[Agent Response]:
<think>
Okay, the user provided their name as Alice and mentioned their email but it's redacted. They want me to say hello. Let me start with a friendly greeting. I should make sure to acknowledge their name and maybe offer further assistance. I need to avoid mentioning the email since it's redacted. Keep it simple and welcoming. Let me check the response again to see if it's appropriate. "Hello Alice! It's nice to meet you. How can I assist you today?" That sounds good. It's polite and opens the door for them to ask for help. I don't mention the email, which is correct. Alright, that should work.
</think>

Hello Alice! It's nice to meet you. How can I assist you today?


> **Notice:** If you inspect the runtime logs or LangSmith tracing, you will see that the model received `alice.secret@example.com` already redacted to `[REDACTED_EMAIL]`, ensuring the sensitive data never leaves your environment!

## 4. Runaway Loops & Budget Guardrails (`ModelCallLimitMiddleware`)

Agents with tools can sometimes fall into infinite reflection loops (repeatedly calling models or tools without reaching a conclusion). This wastes tokens, increases latency, and balloons API bills.

`ModelCallLimitMiddleware` allows you to set a maximum cap on model calls per run.

In [7]:
from langchain.agents.middleware import ModelCallLimitMiddleware

# Cap the model at a maximum of 1 model call per execution run
limit_middleware = ModelCallLimitMiddleware(
    run_limit=1, 
    exit_behavior="end"  # Returns the current state instead of throwing an error
)

# Let's give the agent a tool so it wants to call the model multiple times (1. call tool, 2. analyze tool output)
agent_with_limit = create_agent(
    model=llm,
    tools=[get_stock_price],
    middleware=[limit_middleware]
)

try:
    result = agent_with_limit.invoke({
        "messages": [{"role": "user", "content": "What is the stock price of MSFT?"}]
    })
    print("\nExecution finished successfully. Message list length:", len(result["messages"]))
    for i, msg in enumerate(result["messages"]):
        print(f"Message {i} ({msg.__class__.__name__}): {getattr(msg, 'content', '')[:100]}...")
except Exception as e:
    print(f"\nAgent execution failed with error: {e}")

[Tool] Attempt 4: Fetching price for MSFT...

Execution finished successfully. Message list length: 4
Message 0 (HumanMessage): What is the stock price of MSFT?...
Message 1 (AIMessage): ...
Message 2 (ToolMessage): The price of MSFT is $240.50...
Message 3 (AIMessage): Model call limits exceeded: run limit (1/1)...


## 5. Creating Custom Middleware

If the built-in middlewares do not cover your exact requirements, you can write custom ones. There are two clean approaches:

### Approach A: Decorator-based Middleware (Simple Functions)
Best for basic, single-action hooks. You can decorate simple functions with:
- `@before_agent`: Runs when the agent is first called.
- `@before_model`: Intercepts the state right before contacting the LLM.
- `@after_model`: Intercepts the LLM response.
- `@after_agent`: Runs right before returning results to the user.

In [8]:
from langchain.agents.middleware import before_model

@before_model
def print_state_summary(state, runtime):
    print("\n--- [Custom Middleware] Pre-Model Call ---")
    print(f"Current message count: {len(state['messages'])}")
    # Return None to continue the agent execution unaltered,
    # or return a dict of updates to merge into the agent's state.
    return None

# Create agent with decorated custom middleware
custom_agent = create_agent(
    model=llm,
    middleware=[print_state_summary]
)

custom_agent.invoke({
    "messages": [{"role": "user", "content": "Tell me a one-word joke"}]
})


--- [Custom Middleware] Pre-Model Call ---
Current message count: 1


{'messages': [HumanMessage(content='Tell me a one-word joke', additional_kwargs={}, response_metadata={}, id='e02f098f-c74f-458b-90f3-a2cb81c5e4b4'),
  AIMessage(content='<think>\nOkay, the user asked for a one-word joke. Let me think about how to approach this. First, a one-word joke is tricky because it has to rely on context or a play on words. Maybe a pun? Or something that\'s a word that sounds like another word.\n\nHmm, "merry" and "Christmas" come to mind, but that\'s two words. Wait, maybe a single word that has a double meaning. Like "sigh" or something? Not sure. Let me think of some possibilities.\n\nOh, "awkward" is a word that\'s often used in jokes. Like when you\'re sitting in a chair that\'s shaped like an "awkward" person. No, that\'s not quite. Maybe "chair" itself? If someone sits in a chair and it\'s a play on words. Not quite.\n\nWait, there\'s the classic "Why did the scarecrow win an award? Because he was a straw man." But that\'s not one word. Maybe a single wor

### Approach B: Class-based Middleware (`AgentMiddleware`)
Best for robust, stateful, or reusable logic. By inheriting from `AgentMiddleware`, you can implement any of the lifecycle hooks, and also intercept tool executions with `wrap_tool_call` or model calls with `wrap_model_call`.

In [9]:
from langchain.agents.middleware import AgentMiddleware

class AuditLogMiddleware(AgentMiddleware):
    """Custom middleware that logs every action taken by the agent to a file or stream."""
    
    def before_agent(self, state, runtime):
        print("[Audit Log] Session started.")
        return None
        
    def after_agent(self, state, runtime):
        print("[Audit Log] Session finished.")
        return None
        
    def before_model(self, state, runtime):
        print("[Audit Log] Contacting the LLM...")
        return None

# Use our custom class middleware
audit_agent = create_agent(
    model=llm,
    middleware=[AuditLogMiddleware()]
)

audit_agent.invoke({
    "messages": [{"role": "user", "content": "Hi!"}]
})

[Audit Log] Session started.
[Audit Log] Contacting the LLM...
[Audit Log] Session finished.


{'messages': [HumanMessage(content='Hi!', additional_kwargs={}, response_metadata={}, id='f5363466-4af1-4b3d-8268-9eb4e5701865'),
  AIMessage(content='<think>\nOkay, the user said "Hi!". I need to respond appropriately. Since it\'s a greeting, I should reply with a friendly greeting. Maybe add an emoji to keep it warm. Also, offer help in case they need anything. Keep it concise and welcoming. Let me make sure the tone is positive and approachable. Yeah, that should work.\n</think>\n\nHello! 😊 How can I assist you today? Let me know if you have any questions or need help with anything!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 10, 'total_tokens': 111, 'completion_time': 0.201750709, 'completion_tokens_details': None, 'prompt_time': 0.000309528, 'prompt_tokens_details': None, 'queue_time': 0.049378682, 'total_time': 0.202060237}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand',

## Summary

LangChain's **Agent Middleware** architecture provides a structured, decoupled, and production-ready way to control your LLM agent workflows.

| Feature | Callbacks | Middleware |
| :--- | :--- | :--- |
| **Primary Purpose** | Observability, tracing, and read-only logging | Interception, request/response editing, flow-control |
| **State Access** | Read-only | Read & Write (can modify Agent State) |
| **Flow Interruption**| Cannot stop execution | Can short-circuit, retry, or jump to specific nodes |
| **Ideal for** | LangSmith, prompt/response tracing, cost tracking | PII redactors, auto-retries, rate-limiting, guardrails |